In [1]:
import pandas as pd

url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
url_white = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

red = pd.read_csv(url_red, sep=";")
white = pd.read_csv(url_white, sep=";")

Encode color variable to 0/1, the concat the datasets

In [2]:
red["color"] = 0
white["color"] = 1
wine = pd.concat([red, white], ignore_index=True)
wine = wine.sample(frac=1, random_state=1234).reset_index(drop=True)
wine.shape

(6497, 13)

In [3]:
X = wine.drop("quality", axis=1)
y = wine["quality"]
X.shape, y.shape

((6497, 12), (6497,))

In [4]:
X.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,color
0,6.0,0.33,0.20,1.80,0.031,49.0,159.0,0.99190,3.41,0.53,11.0,1
1,7.2,0.29,0.20,7.70,0.046,51.0,174.0,0.99582,3.16,0.52,9.5,1
2,6.7,0.47,0.29,4.75,0.034,29.0,134.0,0.99056,3.29,0.46,13.0,1
3,7.6,0.43,0.31,2.10,0.069,13.0,74.0,0.99580,3.26,0.54,9.9,0
4,6.8,0.68,0.09,3.90,0.068,15.0,29.0,0.99524,3.41,0.52,11.1,0


In [5]:
from sklearn.model_selection import train_test_split

seed = 1234

# first split test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed,
    stratify=y,
)

# split train / validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=seed,
    stratify=y_train_full,
)

scale X

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# y stays same

In [10]:
from sklearn.neural_network import MLPClassifier

layers_list = [(64, 32), (128, 64), (128, 64, 32)]
lr_list = [0.001, 0.005, 0.01]
alpha_list = [0.0001, 0.001,0.01]
batch_list = [64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train)

                train_acc = model.score(X_train_scaled, y_train)
                val_acc = model.score(X_val_scaled, y_val)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9689504747241467, 'val_acc': 0.6130769230769231}
{'layers': (128, 64, 32), 'lr': 0.01, 'alpha': 0.001, 'batch_size': 128, 'train_acc': 0.9225044906338209, 'val_acc': 0.6023076923076923}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.001, 'batch_size': 64, 'train_acc': 0.9633051064921735, 'val_acc': 0.6015384615384616}
{'layers': (128, 64, 32), 'lr': 0.005, 'alpha': 0.0001, 'batch_size': 64, 'train_acc': 0.9661277906081601, 'val_acc': 0.6015384615384616}
{'layers': (128, 64, 32), 'lr': 0.005, 'alpha': 0.001, 'batch_size': 64, 'train_acc': 0.9630484988452656, 'val_acc': 0.5992307692307692}
{'layers': (128, 64, 32), 'lr': 0.001, 'alpha': 0.0001, 'batch_size': 128, 'train_acc': 0.9758788811906595, 'val_acc': 0.5984615384615385}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9481652553246087, 'val_acc': 0.5953846153846154}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.0001,

Overfitting. Switch to large alpha 

In [13]:
from sklearn.neural_network import MLPClassifier

layers_list = [(64, 32), (128, 64),(256,128)]
lr_list = [0.001, 0.005, 0.01]
alpha_list = [0.01,0.02,0.05]
batch_list = [64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train)

                train_acc = model.score(X_train_scaled, y_train)
                val_acc = model.score(X_val_scaled, y_val)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9689504747241467, 'val_acc': 0.6130769230769231}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9304593276879651, 'val_acc': 0.6130769230769231}
{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9905055170644085, 'val_acc': 0.6092307692307692}
{'layers': (256, 128), 'lr': 0.005, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9550936617911214, 'val_acc': 0.6069230769230769}
{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9863997947138825, 'val_acc': 0.6038461538461538}
{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.02, 'batch_size': 128, 'train_acc': 0.9799846035411856, 'val_acc': 0.6015384615384616}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.02, 'batch_size': 64, 'train_acc': 0.9343084423915833, 'val_acc': 0.5992307692307692}
{'layers': (256, 128), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 64, '

Stable layers and learning rate. Tune alpha and batch size.

In [14]:
from sklearn.neural_network import MLPClassifier

layers_list = [(128, 64)]
lr_list = [0.001]
alpha_list = [0.01,0.02,0.03,0.04,0.05]
batch_list = [16, 32, 64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train)

                train_acc = model.score(X_train_scaled, y_train)
                val_acc = model.score(X_val_scaled, y_val)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9689504747241467, 'val_acc': 0.6130769230769231}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9481652553246087, 'val_acc': 0.5953846153846154}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.03, 'batch_size': 32, 'train_acc': 0.8991531947652041, 'val_acc': 0.5938461538461538}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.05, 'batch_size': 64, 'train_acc': 0.8755452912496793, 'val_acc': 0.5938461538461538}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.02, 'batch_size': 128, 'train_acc': 0.9173723376956633, 'val_acc': 0.5907692307692308}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.8691301000769823, 'val_acc': 0.5907692307692308}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.02, 'batch_size': 32, 'train_acc': 0.9317423659225045, 'val_acc': 0.59}
{'layers': (128, 64), 'lr': 0.001, 'alpha': 0.02, 'batch_size': 64, 'train_acc': 0.9222

Final results:

layers: (128, 64)

learning_rate: 0.001

alpha: 0.01

batch_size: 64

train_acc: 0.9689504747241467

val_acc: 0.6130769230769231

In [8]:
y.value_counts().sort_index()

quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5
Name: count, dtype: int64

Switch to binary y

In [9]:
y_test_cls = (y_test >= 6).astype(int)
y_train_cls = (y_train >= 6).astype(int)
y_val_cls = (y_val >= 6).astype(int)

from sklearn.neural_network import MLPClassifier

layers_list = [(64, 32), (128, 64), (128, 64, 32)]
lr_list = [0.001, 0.005, 0.01]
alpha_list = [0.0001, 0.001,0.01]
batch_list = [64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train_cls)

                train_acc = model.score(X_train_scaled, y_train_cls)
                val_acc = model.score(X_val_scaled, y_val_cls)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9743392353092122, 'val_acc': 0.7953846153846154}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9689504747241467, 'val_acc': 0.7876923076923077}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.001, 'batch_size': 64, 'train_acc': 0.9820374647164486, 'val_acc': 0.7861538461538462}
{'layers': (128, 64, 32), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9561200923787528, 'val_acc': 0.7861538461538462}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.0001, 'batch_size': 64, 'train_acc': 0.9881960482422376, 'val_acc': 0.7853846153846153}
{'layers': (128, 64, 32), 'lr': 0.005, 'alpha': 0.001, 'batch_size': 64, 'train_acc': 0.9881960482422376, 'val_acc': 0.7853846153846153}
{'layers': (64, 32), 'lr': 0.01, 'alpha': 0.0001, 'batch_size': 128, 'train_acc': 0.9676674364896074, 'val_acc': 0.7846153846153846}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.0001, 'batch_size'

In [10]:
layers_list = [(64, 32), (128, 64),(256,128)]
lr_list = [0.001, 0.005, 0.01]
alpha_list = [0.01,0.02,0.05]
batch_list = [64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train_cls)

                train_acc = model.score(X_train_scaled, y_train_cls)
                val_acc = model.score(X_val_scaled, y_val_cls)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9758788811906595, 'val_acc': 0.8}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9743392353092122, 'val_acc': 0.7953846153846154}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9440595329740826, 'val_acc': 0.7915384615384615}
{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9912753400051322, 'val_acc': 0.7915384615384615}
{'layers': (64, 32), 'lr': 0.001, 'alpha': 0.02, 'batch_size': 64, 'train_acc': 0.9343084423915833, 'val_acc': 0.7907692307692308}
{'layers': (256, 128), 'lr': 0.001, 'alpha': 0.05, 'batch_size': 64, 'train_acc': 0.9507313317936874, 'val_acc': 0.7892307692307692}
{'layers': (256, 128), 'lr': 0.005, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9763920964844752, 'val_acc': 0.7884615384615384}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9

In [11]:
layers_list = [(128, 64),(256,128)]
lr_list = [0.005, 0.01]
alpha_list = [0.01,0.02,0.05]
batch_list = [16, 32, 64, 128]

results = []

for layers in layers_list:
    for lr in lr_list:
        for alpha in alpha_list:
            for batch in batch_list:
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    batch_size=batch,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

                model.fit(X_train_scaled, y_train_cls)

                train_acc = model.score(X_train_scaled, y_train_cls)
                val_acc = model.score(X_val_scaled, y_val_cls)

                results.append({
                    "layers": layers,
                    "lr": lr,
                    "alpha": alpha,
                    "batch_size": batch,
                    "train_acc": train_acc,
                    "val_acc": val_acc
                })

results_sorted = sorted(results, key=lambda x: x["val_acc"], reverse=True)

for r in results_sorted:
    print(r)

{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.01, 'batch_size': 64, 'train_acc': 0.9743392353092122, 'val_acc': 0.7953846153846154}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9440595329740826, 'val_acc': 0.7915384615384615}
{'layers': (256, 128), 'lr': 0.005, 'alpha': 0.05, 'batch_size': 128, 'train_acc': 0.9763920964844752, 'val_acc': 0.7884615384615384}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 128, 'train_acc': 0.9689504747241467, 'val_acc': 0.7876923076923077}
{'layers': (128, 64), 'lr': 0.01, 'alpha': 0.01, 'batch_size': 32, 'train_acc': 0.8604054400821144, 'val_acc': 0.786923076923077}
{'layers': (128, 64), 'lr': 0.005, 'alpha': 0.02, 'batch_size': 128, 'train_acc': 0.9720297664870413, 'val_acc': 0.7853846153846153}
{'layers': (256, 128), 'lr': 0.005, 'alpha': 0.02, 'batch_size': 128, 'train_acc': 0.9802412111880934, 'val_acc': 0.7853846153846153}
{'layers': (256, 128), 'lr': 0.005, 'alpha': 0.01, 'batch_size': 128, 't

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score

model = MLPClassifier(
                    hidden_layer_sizes=(256,128),
                    activation='relu',
                    learning_rate_init=0.001,
                    alpha=0.05,
                    batch_size=128,
                    max_iter=1000,
                    random_state=seed,
                    n_iter_no_change=10,
                )

model.fit(X_train_scaled, y_train_cls)

train_acc = model.score(X_train_scaled, y_train_cls)
val_acc = model.score(X_val_scaled, y_val_cls)
precision = precision_score(y_val_cls, model.predict(X_val_scaled))
recall = recall_score(y_val_cls, model.predict(X_val_scaled))
f1 = f1_score(y_val_cls, model.predict(X_val_scaled))

print(train_acc, val_acc, precision, recall, f1)


0.9758788811906595 0.8 0.8246828143021915 0.8687727825030377 0.8461538461538461


Final results:

layers: (256, 128)

learning_rate: 0.001

alpha: 0.05

batch_size: 128

train_acc: 0.9758788811906595

val_acc: 0.8

precision: 0.8246828143021915 

recall: 0.8687727825030377

f1: 0.8461538461538461